In [1]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd
import time
import json
import pandas as pd

In [2]:
load_dotenv()  
def get_token(secret: str, url: str = "https://dep2.simondg.com/auth/login"):
    response = requests.post(url, json={"secret": secret})
    response.raise_for_status() # Raises an error if the request fails
    return response.json()["access_token"]


def get_students_by_subgroup(token: str, subgroup_id: int):
    url = f"https://dep2.simondg.com/students/{subgroup_id}"
    headers = {"Authorization": f"Bearer {token}"}
    
    response = requests.get(url, headers=headers)
    response.raise_for_status()  # Raises an error if the request fails
    
    data = response.json()
    
    return data


secret = os.getenv("SECRET")
token = get_token(secret)

In [3]:
print(get_students_by_subgroup(token, 6094292 ) )  #test

{'students': [{'SUBGROEPID': 6094292, 'SUBGROEPCODE': 'PBA-VG-LAM/AO/2', 'DEELGROEPID': 27519, 'NAAM': 'u_65ce9dc8949af0e64966d5a89b393b51f91dd47d4c84276b3dce3956123a9421', 'EMAIL': 'u_b18885f309bdcae75b20ec3b241f8c4d35cd050966e37ae8707e52fcf4c3aed2'}, {'SUBGROEPID': 6094292, 'SUBGROEPCODE': 'PBA-VG-LAM/AO/2', 'DEELGROEPID': 27519, 'NAAM': 'u_65ce9dc8949af0e64966d5a89b393b51f91dd47d4c84276b3dce3956123a9421', 'EMAIL': 'u_8eb47e148d1cc598f2cff8a49f13c2755f91b6372357446257380209685a62a7'}, {'SUBGROEPID': 6094292, 'SUBGROEPCODE': 'PBA-VG-LAM/AO/2', 'DEELGROEPID': 27519, 'NAAM': 'u_8a3955abeab562db29a9b1a2fd7a3021120324d5da29871dc8c117d9bf348133', 'EMAIL': 'u_2a00feda6868bd0ff18dad547b9b6de4d3495974b06a9a8e3b87eb326c56a1a2'}, {'SUBGROEPID': 6094292, 'SUBGROEPCODE': 'PBA-VG-LAM/AO/2', 'DEELGROEPID': 27519, 'NAAM': 'u_b64c6d4f872f781fc4cf3b6e425d189af5c7447c05cbd36fcbc4f401d287b901', 'EMAIL': 'u_b1aa4c2110b57db4d4c77dc987248d2c84dad58d0855304e54c4847f63cd2620'}, {'SUBGROEPID': 6094292, 'SUBGR

In [4]:
# Your subgroup file
subgroup_file = "../data/unique_classgroups.csv"
df_subgroups = pd.read_csv(subgroup_file, header=None, names=['subgroup_id'])

all_data_json = {"students": []}

MAX_RETRIES = 5
INITIAL_WAIT = 10  # seconds

def fetch_students_with_retry(subgroup_id, token):
    """Fetch students with retry and skip for server errors."""
    retries = 0
    wait = INITIAL_WAIT
    while retries < MAX_RETRIES:
        try:
            students = get_students_by_subgroup(token, str(subgroup_id))
            return students
        except requests.exceptions.HTTPError as e:
            status = e.response.status_code
            if status == 429:
                print(f"⚠️ Rate limit hit for {subgroup_id}, waiting {wait}s...")
                time.sleep(wait)
                retries += 1
                wait *= 2
            elif status == 500:
                print(f"❌ Server error (500) for {subgroup_id} — skipping this subgroup.")
                return []
            else:
                print(f"❌ HTTP {status} for {subgroup_id}: {e}")
                return []
        except Exception as e:
            print(f"⚠️ Error fetching {subgroup_id}: {e}, retrying in {wait}s...")
            time.sleep(wait)
            retries += 1
            wait *= 2

    print(f"❌ Failed to fetch {subgroup_id} after {MAX_RETRIES} retries — skipping.")
    return []

existing_students_csv = "../data/all_students.csv"

df_existing = pd.read_csv(existing_students_csv)
existing_ids = set(df_existing["SUBGROEPID"].astype(str))


for subgroup_id in df_subgroups['subgroup_id'].dropna():
    subgroup_id_str = str(subgroup_id)
    if str(subgroup_id).startswith("EK"):
        continue

    if subgroup_id_str in existing_ids:
        # print(f"⏩ Subgroup {subgroup_id_str} already exists in all_students.csv — skipping fetch.")
        continue


    json_filename = f"../data/classgroups/{subgroup_id}.json"

    if os.path.exists(json_filename):
        with open(json_filename, "r") as f:
            students = json.load(f).get("students", [])
    else:
        print(f"Fetching data for subgroup {subgroup_id}")
        students = fetch_students_with_retry(subgroup_id, token)
        if students:
            with open(json_filename, "w") as f:
                json.dump({"students": students}, f, indent=4)
        # else:
        #     
        #     with open(json_filename, "w") as f:
        #         json.dump({"students": []}, f)

#     all_data_json["students"].extend(students)

# combined_json_path = "../data/classgroups/all_students.json"
# with open(combined_json_path, "w") as f:
#     json.dump(all_data_json, f, indent=4)

#TODO: get new token upon expiration time
print("✅")

C:\Users\Korneel\AppData\Local\Temp\ipykernel_7132\4281852002.py:42: DtypeWarning: Columns (0,2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_existing = pd.read_csv(existing_students_csv)


✅


In [5]:
import json
import os

input_folder = "../data/classgroups"
output_file = "../data/classgroups/all_students.json"

all_students = []

def extract_students(data):
    if isinstance(data, list):
        if data and isinstance(data[0], dict) and "EMAIL" in data[0]:
            return data
        return []
    if isinstance(data, dict):
        for key, value in data.items():
            if key.strip().lower() == "students":
                if isinstance(value, list):
                    return value
                elif isinstance(value, dict):
                    inner = extract_students(value)
                    if inner:
                        return inner
        for v in data.values():
            inner = extract_students(v)
            if inner:
                return inner
    return []


for filename in os.listdir(input_folder):
    if not filename.endswith(".json"):
        continue

    path = os.path.join(input_folder, filename)
    try:
        with open(path, "r", encoding="utf-8-sig") as f:
            data = json.load(f)
    except Exception as e:
        print(f"⚠️ Skipping {filename}: {e}")
        continue

    students = extract_students(data)
    if students:
        print(f"✅ {filename}: found {len(students)} students")
        all_students.extend(students)
    else:
        print(f"❌ {filename}: no students found")

print(f"\n📊 Total students collected: {len(all_students)}")

# ✅ keep everyone, duplicates and all
merged = {"students": all_students}

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(merged, f, indent=4, ensure_ascii=False)

print(f"💾 Saved {len(all_students)} students to {output_file}")


❌ 4889294.json: no students found
❌ 4993432.json: no students found
❌ 4993473.json: no students found
❌ 5124387.json: no students found
❌ 5278847.json: no students found
❌ 5278877.json: no students found
❌ 5279133.json: no students found
❌ 5279163.json: no students found
❌ 5279419.json: no students found
❌ 5279449.json: no students found
❌ 5303876.json: no students found
❌ 5303938.json: no students found
❌ 5316136.json: no students found
❌ 5352461.json: no students found
❌ 5365421.json: no students found
❌ 5406186.json: no students found
❌ 5425818.json: no students found
❌ 5452209.json: no students found
❌ 5452495.json: no students found
❌ 5501899.json: no students found
❌ 5516218.json: no students found
❌ 5567109.json: no students found
❌ 5651631.json: no students found
❌ 5651632.json: no students found
❌ 5651633.json: no students found
❌ 5677872.json: no students found
❌ 5690655.json: no students found
❌ 5717669.json: no students found
✅ 5717692.json: found 42 students
✅ 5717823.json

In [ ]:

# Path to your JSON file
json_file = "../data/classgroups/all_students.json"

# Load JSON data
with open(json_file, "r") as f:
    data = json.load(f)

# Extract 'students' list
students_list = data.get("students", [])

# Convert to DataFrame
students_df = pd.DataFrame(students_list)

# Path to output file
output_file = "../data/all_students.csv"

# If the file exists, load and merge to keep only unique rows
if os.path.exists(output_file):
    existing_df = pd.read_csv(output_file)
    students_df = pd.concat([existing_df, students_df]).drop_duplicates()

# Save the deduplicated result
students_df.to_csv(output_file, index=False)

# Also save a copy in the classgroups folder
students_df.to_csv("../data/classgroups/all_students.csv", index=False)

print(f"✅ Saved {len(students_df)} unique student rows to {output_file}")

In [7]:
students_df.head(10)

,SUBGROEPID,SUBGROEPCODE,DEELGROEPID,NAAM,EMAIL
0,5717692,PBA-SO/LOBR/2A,26933,u_3b2a15b5f0c32963f61e6f19223a97a6ba8de180fc2a...,u_698418af5bb8514dad57f8dec9530a6f060b318ac154...
1,5717692,PBA-SO/LOBR/2A,26933,u_a02a0acb94aac59175ff72e23377d4bd66a11529e5dc...,u_56db3d415cdf0980ab3c5e666adad5175ff1b660c3b7...
2,5717692,PBA-SO/LOBR/2A,26933,u_5573097a6230aedef054da9d0faa056d32dddef571e7...,u_f43e5a6928057557eca3b23f7652e22d6d4307bc92d1...
3,5717692,PBA-SO/LOBR/2A,26933,u_9e1425ffffa163541d16b5d2b0100b62155118e98ea8...,u_57c767e9b97d5b4d5dc485127229f6fa0b48aed1d30e...
4,5717692,PBA-SO/LOBR/2A,26933,u_1547049264404940adf4ff35ea1791f3c0accee5bfea...,u_c7460742de674a83680ac297f66296046186d229a467...
5,5717692,PBA-SO/LOBR/2A,26933,u_29ceec60066842340b4010a7f0b3edeb14876bd871fb...,u_989ea5845559463fdcc827b78220ef7a6d4b3525329b...
6,5717692,PBA-SO/LOBR/2A,26933,u_2be074ab1d08cf124cca95f35f239d87313d0f26371a...,u_d2a5da0c67ca51970f3cc23789d0bf6ec28381ca1dfa...
7,5717692,PBA-SO/LOBR/2A,26933,u_1196d1e0b66f7663135b76e219c723364f7a5aa1b4c4...,u_dc8e5c98834c660f03145f074d410b28e7d737f5d657...
8,5717692,PBA-SO/LOBR/2A,26933,u_f33482d3f22283c92996045b7d2de257e00641b4f276...,u_dfa92702f03c786045582a5ed83ff200dbc658638c36...
9,5717692,PBA-SO/LOBR/2A,26933,u_2f68fc13ed3237ff1993f829173f72243ffba3f44cfc...,u_c5e90bd6e145dbcb3a9cda382504cf90ee45780bd766...
